# PAVIA4 suite — DART + DARTS (+CFAR) + baselines, noise-before, configurable
Every knob lives in the CONFIG cell (written to `p4_config.json`; the engine
reads only that + `DET`): **target** (`target_cls: null` = protocol bitumen
foreign signature, theta .15 paper cell; `target_cls: 5` = painted metal
sheets, theta .075), model configs (DART width, full DARTS architecture),
training configs (lr / wd / clip / batch / epochs / rhos / seeds), CFAR
(lambda, windows, guard). Frozen recipe defaults: noise BEFORE the std front
(no floors), DART Adam wd=0 clip 1.0, DARTS published AdamW wd 1e-5.
Every eval records plain AND CFAR metrics; fp16 snapshots at every eval,
best+final states, final raw score maps saved. Restart-safe (done keys skip).

In [ ]:
!git clone -b camera-ready --depth 1 https://github.com/michaelpiro/final-paper-experiment.git repo
%cd repo
import os, torch
print('device:', 'cuda' if torch.cuda.is_available() else 'cpu')
assert os.path.exists('repro/data/pavia-u.mat'), 'missing data'

In [ ]:
# ======================= CONFIG — edit me =======================
CONFIG = {
  # ---- target ----
  'target_cls': None,     # None = protocol bitumen signature | 5 = metal sheets
  'theta': 0.15,          # bitumen paper cell .15 | metal cell .075
  'seeds': [42, 43],
  'ckpt_every': 200,      # eval + fp16 snapshot cadence (epochs)
  # ---- DART (noise-before, std front) ----
  'dart': {
    'rhos': [0.001, 0.01],
    'epochs': 30000,
    'hidden': [128],      # net width (paper: 128 on pavia)
    'activation': 'relu',
    'lr': 5e-4, 'weight_decay': 0.0, 'grad_clip': 1.0, 'batch_size': 512,
  },
  # ---- DARTS (published arch/optimizer; edit freely) ----
  'darts': {
    'rhos': [0.03, 0.5],
    'epochs': 10000,
    'd_lat': 16, 'K': 7, 'enc_hidden': [64, 32], 'score_hidden': [128],
    'activation': 'relu',
    'lr': 3e-4, 'weight_decay': 1e-5, 'grad_clip': 1.0, 'batch_size': 512,
  },
  # ---- CFAR ----
  'cfar': {'lam': 0.1, 'dart_win': 5, 'darts_win': None,  # None -> k
           'guard': 1},
}
# ================================================================
import json
json.dump(CONFIG, open('p4_config.json', 'w'), indent=1)
print(json.dumps(CONFIG, indent=1))

In [ ]:
%%writefile run_p4.py
"""pavia4 engine — DART + DARTS, noise-before, std front (no floors).
All configuration from p4_config.json (see CONFIG cell); env DET=dart|darts
selects the detector. Outputs: results_p4.json (curves, plain+CFAR metrics,
best/final) and ckpt_p4/<key>/ (fp16 snaps + model states + score maps)."""
import json
import os
import sys
import time

sys.path.insert(0, os.getcwd())

import numpy as np
import torch
from tqdm import tqdm

from repro import scenes
from repro.scenes import pavia_protocol as PP
from repro.protocols.spatial import load_cfg
from repro.core.data import Whitening, plant_targets, extract_neighborhoods
from repro.core.metrics import auc_safe, dr_at_fpr
from repro.core.seeding import seed_all
from repro.core.models import ScoreNet
from repro.models.darts.model import DARTS, _NeighborDenoiser

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.set_grad_enabled(True)
CFG = json.load(open('p4_config.json'))
DET = os.environ.get('DET', 'dart')
DCFG = CFG[DET]
THETA = float(CFG['theta'])
TARGET_CLS = CFG['target_cls']
SEEDS = [int(s) for s in CFG['seeds']]
CKPT_EVERY = int(CFG['ckpt_every'])
EPOCHS = int(DCFG['epochs'])
TAG = 'p4' if TARGET_CLS is None else f'p4cls{TARGET_CLS}'
OUT_JSON = 'results_p4.json'
CKPT_ROOT = 'ckpt_p4'

SP_CFG = load_cfg()
os.makedirs(CKPT_ROOT, exist_ok=True)
_SC = {}


def get_scene():
    if not _SC:
        sc = scenes.build('pavia4', SP_CFG)
        k = int(SP_CFG['k'])
        for key, flat, shape in (('_te_nbr', sc['te'], sc['te_shape']),
                                 ('_tr_nbr', sc['tr'], sc['tr_shape'])):
            img = torch.tensor(np.asarray(flat, np.float32)
                               .reshape(*shape, -1))
            _, nbr = extract_neighborhoods(img, k)
            sc[key] = nbr.numpy()
        if TARGET_CLS is not None:
            sc['sig_use'] = PP.foreign_signature(
                sc['data'], sc['gt'], sc['te'],
                cls=int(TARGET_CLS)).astype(np.float32)
        else:
            sc['sig_use'] = np.asarray(sc['sig'], np.float32)
        _SC.update(sc)
    return _SC


def std_front(tr):
    X = np.asarray(tr, np.float64)
    return Whitening(X.mean(0).astype(np.float32),
                     np.diag(1.0 / X.std(0)).astype(np.float32))


def metrics(y, T):
    return (float(auc_safe(y, T)),
            float(dr_at_fpr(y, T, fpr_list=(0.05,))['0.05']))


def cfar(T, shape, win, guard):
    return DARTS.local_moment_normalize(
        T, shape, win, guard=guard, cfar_lam=float(CFG['cfar']['lam']))


def run_one(rho, seed):
    key = f'{TAG}_{DET}_r{rho}_s{seed}'
    res = json.load(open(OUT_JSON)) if os.path.exists(OUT_JSON) else {}
    if key in res:
        print('skip (done):', key); return
    t0 = time.time()
    sc = get_scene()
    tr, te, s = sc['tr'], sc['te'], sc['sig_use']
    D = tr.shape[1]
    W = std_front(tr)
    sigma = float(np.sqrt(rho * np.asarray(tr, np.float64).var(0).mean()))
    planted, labels, _ = plant_targets(
        te, s, THETA, float(SP_CFG['target_fraction']), model='additive',
        seed=seed, spatial_shape=sc['te_shape'],
        edge_guard=int(SP_CFG['edge_guard']))
    planted = planted.astype(np.float32)
    y = np.asarray(labels)
    seed_all(seed)
    if DET == 'dart':
        net = ScoreNet(D, list(DCFG['hidden']), DCFG['activation'],
                       whitening=W).to(DEVICE)
        opt = torch.optim.Adam(net.parameters(), lr=float(DCFG['lr']),
                               weight_decay=float(DCFG['weight_decay']))
        cwin = int(CFG['cfar']['dart_win'])
    else:
        net = _NeighborDenoiser(D, int(DCFG['d_lat']), int(DCFG['K']),
                                list(DCFG['enc_hidden']),
                                list(DCFG['score_hidden']),
                                float(np.sqrt(SP_CFG['darts']['dsm_sigma_rho'])),
                                DCFG['activation'], W).to(DEVICE)
        opt = torch.optim.AdamW(net.parameters(), lr=float(DCFG['lr']),
                                weight_decay=float(DCFG['weight_decay']))
        cwin = int(CFG['cfar']['darts_win'] or SP_CFG['k'])
    batch = int(DCFG['batch_size'])
    clip = float(DCFG['grad_clip'])
    cguard = int(CFG['cfar']['guard'])
    X = torch.tensor(np.asarray(tr, np.float32), device=DEVICE)
    N = (torch.tensor(np.asarray(sc['_tr_nbr'], np.float32), device=DEVICE)
         if DET == 'darts' else None)
    gen = torch.Generator(device=DEVICE); gen.manual_seed(97 * seed)
    P = len(X)
    rundir = os.path.join(CKPT_ROOT, key)
    os.makedirs(rundir, exist_ok=True)

    def score_all(pix, nbr):
        out = []
        with torch.no_grad():
            for i in range(0, len(pix), 1024):
                p = torch.tensor(np.asarray(pix[i:i+1024], np.float32),
                                 device=DEVICE)
                if DET == 'darts':
                    nb = torch.tensor(np.asarray(nbr[i:i+1024], np.float32),
                                      device=DEVICE)
                    out.append(net(p, nb).cpu().numpy())
                else:
                    out.append(net(p).cpu().numpy())
        return np.concatenate(out, 0)

    def detector_T():
        z_tr = score_all(tr, sc['_tr_nbr'])
        z_te = score_all(planted, sc['_te_nbr'])
        zb = z_tr.mean(0)
        C = np.cov(z_tr, rowvar=False)
        return -((z_te - zb) @ s) / np.sqrt(float(s @ C @ s))

    curve = []
    best = {'auc': -1.0}
    best_state = None
    bar = tqdm(range(1, EPOCHS + 1), desc=key, ncols=130, mininterval=5.0,
               file=sys.stdout, ascii=True)
    for ep in bar:
        net.train()
        perm = torch.randperm(P, generator=gen, device=DEVICE)
        for i in range(0, P, batch):
            sel = perm[i:i + batch]
            eps = torch.randn((len(sel), D), generator=gen,
                              device=DEVICE) * sigma
            psi = net(X[sel] + eps, N[sel]) if DET == 'darts' \
                else net(X[sel] + eps)
            loss = ((psi + eps / sigma ** 2) ** 2).sum(-1).mean()
            opt.zero_grad(); loss.backward()
            if clip:
                torch.nn.utils.clip_grad_norm_(net.parameters(), clip)
            opt.step()
        if ep % CKPT_EVERY == 0 or ep == EPOCHS:
            net.eval()
            T = detector_T()
            Tc = cfar(T, sc['te_shape'], cwin, cguard)
            auc, pd05 = metrics(y, T)
            auc_c, pd05_c = metrics(y, Tc)
            curve.append({'epoch': ep, 'auc': round(auc, 4),
                          'pd05': round(pd05, 4),
                          'auc_cfar': round(auc_c, 4),
                          'pd05_cfar': round(pd05_c, 4)})
            if auc > best['auc']:
                best = dict(curve[-1]); best['auc'] = auc
                best_state = {k_: v.cpu().clone()
                              for k_, v in net.state_dict().items()}
            torch.save({'w16': {k_: v.half().cpu() for k_, v in
                                net.state_dict().items()}, 'epoch': ep},
                       os.path.join(rundir, f'snap_{ep:06d}.pt'))
            bar.set_postfix_str(
                f'loss={float(loss.detach()):.3g} auc={auc:.3f} '
                f'cfar={auc_c:.3f} best={best["auc"]:.3f}@{best["epoch"]}')
    bar.close()
    net.eval()
    T = detector_T()
    Tc = cfar(T, sc['te_shape'], cwin, cguard)
    np.savez_compressed(os.path.join(rundir, 'scores_final.npz'),
                        T=T, T_cfar=Tc, labels=y)
    torch.save({'net_final': {k_: v.cpu() for k_, v in
                              net.state_dict().items()},
                'net_best': best_state, 'best': best, 'rho': rho,
                'seed': seed, 'theta': THETA, 'target_cls': TARGET_CLS,
                'sigma_raw': sigma, 'det_cfg': DCFG},
               os.path.join(rundir, 'model.pt'))
    out = {'det': DET, 'tag': TAG, 'rho': rho, 'seed': seed,
           'theta': THETA, 'target_cls': TARGET_CLS,
           'epochs_done': EPOCHS, 'best': best, 'final': curve[-1],
           'curve': curve, 'sigma_raw': round(sigma, 1), 'cfar_win': cwin,
           'det_cfg': DCFG, 'sec': round(time.time() - t0)}
    res = json.load(open(OUT_JSON)) if os.path.exists(OUT_JSON) else {}
    res[key] = out
    json.dump(res, open(OUT_JSON, 'w'), indent=1)
    print(f'[{key}] best_auc={best["auc"]:.3f}@{best["epoch"]} '
          f'final auc={curve[-1]["auc"]} cfar={curve[-1]["auc_cfar"]} '
          f'({out["sec"]}s)', flush=True)


if __name__ == '__main__':
    done = set(json.load(open(OUT_JSON)).keys()) if os.path.exists(OUT_JSON) else set()
    tasks = [(float(r), sd) for r in DCFG['rhos'] for sd in SEEDS
             if f'{TAG}_{DET}_r{float(r)}_s{sd}' not in done]
    print(f'{len(tasks)} tasks [{DET}] tag={TAG} theta={THETA}, '
          f'{EPOCHS} ep, device={DEVICE}', flush=True)
    get_scene()
    for r, sd in tasks:
        run_one(r, sd)
    print('ALL DONE', flush=True)

In [ ]:
# ---- DART on pavia4 ----
import os
os.environ['DET'] = 'dart'
!python run_p4.py

In [ ]:
# ---- DARTS on pavia4 (plain + CFAR metrics both recorded) ----
import os
os.environ['DET'] = 'darts'
!python run_p4.py

In [ ]:
# ---- Baselines: AMF-global / AMF-local / GMM-Levin / LRao (val-ES) ----
import json, os
import numpy as np, torch
from run_p4 import get_scene, SP_CFG, THETA, TARGET_CLS, TAG
from repro.core.data import plant_targets
from repro.core.metrics import auc_safe, dr_at_fpr
from repro.protocols.spatial import _windows
from repro.models.classical import AMF, AMFLocal, GMMLevin
from repro.models.lrao import LRao

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
CFG = json.load(open('p4_config.json'))
SEEDS_B = [int(s) for s in CFG['seeds']]

sc = get_scene()
tr, te, shape = sc['tr'], sc['te'], sc['te_shape']
sig = sc['sig_use']
amf = AMF(SP_CFG).fit(tr)
lev = GMMLevin(SP_CFG).fit(tr)
amf_local = AMFLocal(SP_CFG)
wA = amf_local.resolved_window(tr.shape[1])
_, nbr_amf = _windows(te, shape, wA, DEVICE)
lrao_cfg = dict(SP_CFG['lrao'])
w = (SP_CFG.get('net_width') or {}).get('pavia4')
if w: lrao_cfg['hidden'] = [int(w)]

res = {}
for seed in SEEDS_B:
    planted, labels, _ = plant_targets(
        te, sig, THETA, float(SP_CFG['target_fraction']), model='additive',
        seed=seed, spatial_shape=shape, edge_guard=int(SP_CFG['edge_guard']))
    planted = planted.astype(np.float32); y = np.asarray(labels)
    lrao = LRao(lrao_cfg).fit(tr, seed, DEVICE,
                              run_dir=f'ckpt_p4/lrao_{TAG}_s{seed}')
    scores = {
        'AMF-global': amf.score(planted, sig),
        'AMF-local':  amf_local.score(planted, nbr_amf, sig, device=DEVICE),
        'GMM-Levin':  lev.score(planted, sig),
        'LRao':       lrao.score(planted, tr[lrao.fit_idx], sig),
    }
    np.savez_compressed(f'ckpt_p4/baseline_scores_{TAG}_s{seed}.npz',
                        labels=y, **scores)
    for name, T in scores.items():
        auc = float(auc_safe(y, T))
        pd05 = float(dr_at_fpr(y, T, fpr_list=(0.05,))['0.05'])
        res.setdefault(name, {})[f's{seed}'] = {'auc': round(auc, 4),
                                                'pd05': round(pd05, 4)}
        print(f'[{name} s{seed}] auc={auc:.3f} pd05={pd05:.3f}', flush=True)
for name, d in res.items():
    d['auc_mean'] = round(float(np.mean(
        [v['auc'] for k, v in d.items() if k.startswith('s')])), 4)
json.dump(res, open(f'results_p4_baselines_{TAG}.json', 'w'), indent=1)
print(json.dumps(res, indent=1))

In [ ]:
# ---- Summary table ----
import json, os
import numpy as np
rows = {}
if os.path.exists('results_p4.json'):
    for k, v in json.load(open('results_p4.json')).items():
        lab = f"{v['det'].upper()} rho={v['rho']}"
        rows.setdefault((v['tag'], lab), []).append(
            (v['best']['auc'], v['final']['auc'], v['final']['auc_cfar']))
print(f"{'tag':<10} {'model':<18} {'best_auc':>9} {'final':>7} {'final_cfar':>11}")
for (tag, lab), vals in sorted(rows.items()):
    b, f, c = (np.mean([x[i] for x in vals]) for i in range(3))
    print(f'{tag:<10} {lab:<18} {b:>9.3f} {f:>7.3f} {c:>11.3f}')
import glob
for p in glob.glob('results_p4_baselines_*.json'):
    tag = p.split('baselines_')[1][:-5]
    for name, d in json.load(open(p)).items():
        print(f'{tag:<10} {name:<18} {d["auc_mean"]:>9.3f}')

In [ ]:
# ---- Archive EVERYTHING (results + checkpoints + score maps) ----
import shutil, os, glob
files_ = ' '.join(['results_p4.json', 'p4_config.json'] +
                  glob.glob('results_p4_baselines_*.json'))
os.system(f'zip -q -r p4_results.zip {files_} ckpt_p4')
print(os.path.getsize('p4_results.zip')/1e6, 'MB')
from google.colab import files
shutil.copy('p4_results.zip', 'p4_results_dl.zip')   # copy-then-download
files.download('p4_results_dl.zip')